# Lab 08: How to improve retrieval with hybrid search and reranking

**CPU | API key: `OPENAI_API_KEY` (embedding and chat calls) | In class**

## Goal

Add keyword search and a reranker to the vector search, and decide from a question-by-question comparison whether
each change helps and what it costs. The moment that teaches it: hybrid search, a common design, does not beat the
vector search on these questions and moves a few the wrong way. The reranker fixes one question worded unlike its
answer, adds a model call to every question, and sometimes names a chunk it was never given.

## What you will learn

- How BM25 scores a chunk by the words it shares with a question, by hand and with `bm25s`.
- How reciprocal rank fusion combines two ranked lists without adding scores on different scales.
- How to compare a change with the baseline question by question in a paired table, and what a sign test says.
- How to rerank candidates with a chat model and check the returned IDs in code.
- How to price a retrieval stage in tokens, dollars and seconds.
- How to write a decision from the paired results and the cost.

**In class:** Parts A to D, and the decision table at the end of Part G (*Write the decision*). Parts E to H are
self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-18
OPENAI_API_KEY    found in .env


This lab uses `embed_model` and `rerank_model` from the lines above, and `JSON_MODE`, `TOKEN_ENCODING` and the chat
prices from `models.env`. It embeds the corpus once and makes about 40 rerank calls of about 1,000 tokens each, well
under one US cent on `gpt-4o-mini`. The saved outputs and every count in the text come from `gpt-4o-mini`; the README
says what changed with `gpt-6-luna`. Your own run will differ a little: model replies vary from run to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with the text, not every number.

The baseline is Lab 05's vector store. Keyword search, fusion and the comparisons are plain Python with the `bm25s`
package: LangChain's BM25 and ensemble retrievers sit in its legacy packages, which these labs do not install, and in
production hybrid search usually comes from the search engine or vector database. The reranker's model comes from
LangChain.

---

## Part A: Score the five policies with BM25 by hand

Part A builds BM25, the classic keyword ranking, on the five policies, so you can see what it counts.

In Lab 01 the vector search ranked the policies H01, H04, H02 for *When does the password recovery email stop working
for Atlas?* The answer is in H02 (*Atlas reset links expire after 30 minutes*). Would a ranking by shared words do
better? Start with the words. `terms()` lowercases a text and keeps its words:

In [3]:
import json, math, re
from collections import Counter

def terms(text):
    return re.findall(r"\w+", text.lower())

policies = json.loads((ROOT / "data" / "support-case.json").read_text(encoding="utf-8"))
recovery = "When does the password recovery email stop working for Atlas?"
print("question:", terms(recovery))
print("H02:     ", terms(policies[1]["text"]))

question: ['when', 'does', 'the', 'password', 'recovery', 'email', 'stop', 'working', 'for', 'atlas']
H02:      ['atlas', 'reset', 'links', 'expire', 'after', '30', 'minutes', 'request', 'a', 'new', 'link', 'if', 'the', 'old', 'link', 'has', 'expired']


The question says *stop working*; H02 says *expire*. To a word count those are different words, and so are
*password* and *passwords*.

**BM25** ([Okapi BM25](https://en.wikipedia.org/wiki/Okapi_BM25)) adds up a score for each question word a document
contains, from three ideas:

- **Rare words count more.** The inverse document frequency (IDF) is large for a word few documents contain.
- **Repeats count less and less.** A word that appears twice counts more than once, but not twice as much (`k1`).
- **Short documents count more.** A match in a short document says more than one in a long document (`b`).

`k1=1.2` and `b=0.75` are the usual values. In plain Python:

In [4]:
def bm25(question, texts, k1=1.2, b=0.75):
    counts = [Counter(terms(text)) for text in texts]
    lengths = [sum(count.values()) for count in counts]
    n, average = len(texts), sum(lengths) / len(texts)
    scores = []
    for count, length in zip(counts, lengths):
        score = 0
        for word in set(terms(question)):
            df = sum(word in other for other in counts)          # documents that contain the word
            idf = math.log(1 + (n - df + 0.5) / (df + 0.5))      # rare word, large weight
            tf = count[word]                                      # times it appears in this document
            score += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * length / average))
        scores.append(score)
    return scores

Score the five policies for the recovery question, and print the words each one shares with it:

In [5]:
scores = bm25(recovery, [policy["text"] for policy in policies])
for score, policy in sorted(zip(scores, policies), key=lambda pair: -pair[0]):
    shared = sorted(set(terms(recovery)) & set(terms(policy["text"])))
    print(f"{policy['source_id']}  {score:.3f}  shares {shared}")

H01  1.705  shares ['atlas', 'email']
H02  1.519  shares ['atlas', 'the']
H04  1.412  shares ['password']
H05  0.302  shares ['atlas']
H03  0.293  shares ['atlas']


BM25 ranks H01 first too: it shares *atlas* and *email*. H02 is second only because it contains *the*, which no other policy contains, so in five documents it counts as rare. The words that carry the meaning, *stop working* and *expire*, do not match.

Keyword search finds exact words, such as names, codes and numbers, and misses a paraphrase; the vector search finds
meaning and can miss a rare exact word. Hybrid search runs both and merges the lists. Whether that helps on your
questions is what this lab measures.

[`bm25s`](https://github.com/xhluca/bm25s) is a maintained, fast BM25 package. Its defaults drop English stop words
such as *the*, ignore one-letter words and use `k1=1.5`, so set it up like the hand version and compare the scores:

In [6]:
import bm25s

WORDS = r"(?u)\b\w+\b"                       # every word, as terms() keeps them
tokens = bm25s.tokenize([policy["text"] for policy in policies], token_pattern=WORDS, stopwords=None,
                        show_progress=False)
index = bm25s.BM25(k1=1.2, b=0.75)
index.index(tokens, show_progress=False)

question_words = bm25s.tokenize([recovery], token_pattern=WORDS, stopwords=None, show_progress=False,
                                return_ids=False)[0]
for policy, mine, theirs in zip(policies, scores, index.get_scores(question_words)):
    print(f"{policy['source_id']}  by hand {mine:.3f}  bm25s {theirs:.3f}  ratio {mine / theirs:.2f}")

H01  by hand 1.705  bm25s 0.775  ratio 2.20
H02  by hand 1.519  bm25s 0.690  ratio 2.20
H03  by hand 0.293  bm25s 0.133  ratio 2.20
H04  by hand 1.412  bm25s 0.642  ratio 2.20
H05  by hand 0.302  bm25s 0.137  ratio 2.20


Every ratio is 2.20, `k1 + 1`: the same order on a different scale. `bm25s` uses the variant of Lucene, the engine inside Elasticsearch, which leaves out `(k1 + 1)`.

### Checkpoint A

| You should see | Our run |
|---|---|
| BM25 by hand | H01 1.705, H02 1.519, H04 1.412, H05 0.302, H03 0.293; H02 shares `['atlas', 'the']` |
| `bm25s` | every ratio 2.20; H01 0.775, H02 0.690 |

---

## Part B: Search the corpus with BM25

Part B runs BM25 over the whole corpus and sets it beside the vector search on the 30 development questions.

First the baseline: Lab 05's corpus and vector store, from `shared/lc_pipeline.py`.

In [7]:
from shared import lc_pipeline

corpus = lc_pipeline.load_corpus()
store = lc_pipeline.build_store(lc_pipeline.to_documents(corpus), settings)
questions = json.loads((ROOT / "data" / "development-questions.json").read_text(encoding="utf-8"))
print(len(corpus), "records in the store |", len(questions), "questions")

277 records in the store | 30 questions


Now a BM25 index of the same 277 texts. With the chunk IDs as `corpus`, `retrieve()` returns the top `k` IDs:

In [8]:
bm25_corpus = bm25s.BM25(k1=1.2, b=0.75, corpus=[record["chunk_id"] for record in corpus])
corpus_tokens = bm25s.tokenize([record["text"] for record in corpus], token_pattern=WORDS, stopwords=None,
                               show_progress=False)
bm25_corpus.index(corpus_tokens, show_progress=False)
print("distinct words in the index:", len(corpus_tokens.vocab))

def keyword_search(question, k=10):
    tokens = bm25s.tokenize([question], token_pattern=WORDS, stopwords=None, show_progress=False)
    ids, _ = bm25_corpus.retrieve(tokens, k=k, show_progress=False)
    return [str(chunk_id) for chunk_id in ids[0]]

def vector_search(question, k=10):
    return [document.id for document, _ in lc_pipeline.retrieve(store, question, k=k)]

distinct words in the index: 4083


Both return ten IDs: the reranker in Part E needs more candidates than the three an answer reads. Try the one
development question the vector search ranks wrong at the top, *Who was the Super Bowl 50 MVP?* (answer in
paragraph `S00-P02`):

In [9]:
mvp = "Who was the Super Bowl 50 MVP?"
print("vector:", vector_search(mvp)[:3])
print("BM25:  ", keyword_search(mvp)[:3])
print("chunks that contain 'MVP':", [r["chunk_id"] for r in corpus if "mvp" in terms(r["text"])])

vector: ['S00-P00-C00', 'S00-P02-C00', 'S00-P01-C00']
BM25:   ['S00-P02-C00', 'S00-P00-C01', 'S00-P00-C00']
chunks that contain 'MVP': ['S00-P01-C00', 'S00-P02-C00']


The vector search puts the game summary first and the answer `S00-P02-C00` second; BM25 puts the answer first. *MVP* is in only two of the 277 chunks, a rare word with a large IDF: exact terms are where keyword search wins. Now all 30 questions. For each list, keep the rank of the first chunk from a known source (`None` when
none of the ten is); hit@1 and hit@3 come from those ranks. Both searches are timed for Part F:

In [10]:
import time

src = {record["chunk_id"]: record["source_id"] for record in corpus}

def first_hit(ids, question):
    for rank, chunk_id in enumerate(ids, start=1):
        if src[chunk_id] in question["known_support"]:
            return rank
    return None

def hits(ranks, k):
    return [rank is not None and rank <= k for rank in ranks]

start = time.perf_counter()
dense_hits = [lc_pipeline.retrieve(store, q["question"], k=10) for q in questions]
vector_seconds = time.perf_counter() - start
start = time.perf_counter()
bm25_lists = [keyword_search(q["question"]) for q in questions]
bm25_seconds = time.perf_counter() - start

dense_lists = [[document.id for document, _ in found] for found in dense_hits]
dense_ranks = [first_hit(ids, q) for ids, q in zip(dense_lists, questions)]
bm25_ranks = [first_hit(ids, q) for ids, q in zip(bm25_lists, questions)]
for name, ranks in [("vector", dense_ranks), ("BM25", bm25_ranks)]:
    print(f"{name:6}  hit@1 {sum(hits(ranks, 1))} of 30 | hit@3 {sum(hits(ranks, 3))} of 30")

vector  hit@1 29 of 30 | hit@3 30 of 30
BM25    hit@1 22 of 30 | hit@3 28 of 30


The questions where the two ranks differ:

In [11]:
for q, dense_rank, bm25_rank in zip(questions, dense_ranks, bm25_ranks):
    if dense_rank != bm25_rank:
        print(f"vector {dense_rank} | BM25 {bm25_rank} | {q['question'].strip()}")

vector 2 | BM25 1 | Who was the Super Bowl 50 MVP?
vector 1 | BM25 8 | In what country is Normandy located?
vector 1 | BM25 None | What is the original meaning of the word Norman?
vector 1 | BM25 2 | When was the Latin version of the word Norman first recorded?
vector 1 | BM25 3 | In what year did Nikola Tesla emigrate to the United States?
vector 1 | BM25 2 | By what main attribute are computational problems classified utilizing computational complexity theory?
vector 1 | BM25 3 | What measure of a computational problem broadly defines the inherent difficulty of the solution?
vector 1 | BM25 2 | What method is used to intuitively assess or quantify the amount of resources required to solve a computational problem?
vector 1 | BM25 2 | What field of computer science analyzes the resource requirements of a specific algorithm isolated unto itself within a given problem?


BM25 alone is worse: hit@1 22 of 30 against 29, hit@3 28 against 30. It ranks the MVP answer higher, and every other question that moved went down; for the Norman question the answering chunk is not in its top ten at all (`None`). Hybrid search hopes to keep the vector search's results and add BM25's one win.

### Checkpoint B

| You should see | Our run |
|---|---|
| Index | 277 records; 4,083 distinct words |
| The MVP question | vector `S00-P00-C00`, `S00-P02-C00`, `S00-P01-C00`; BM25 `S00-P02-C00` first |
| 30 questions | vector hit@1 29, hit@3 30; BM25 hit@1 22, hit@3 28; nine questions with different ranks |

---

## Part C: Fuse the two lists with reciprocal rank fusion

Part C merges the vector list and the BM25 list into one ranking.

Adding the scores would not work: BM25 gave H01 1.705 in Part A, while a cosine similarity is at most 1, so BM25's
larger numbers would decide. **Reciprocal rank fusion** (RRF, [Cormack and others,
2009](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)) uses only the ranks. Each list gives a chunk
`1 / (c + rank)`, and the contributions are added; a list without the chunk adds nothing. `c = 60` comes from that
paper. `c` and each list's depth (10 here) are fixed before measuring: tuning them until the 30 questions look better
would measure the tuning.

A keyword list returns B, A; a vector list returns A, C, B:

In [12]:
keyword = ["B", "A"]
vector = ["A", "C", "B"]
fused = {}
for ranked in (keyword, vector):
    for rank, item in enumerate(ranked, start=1):
        fused[item] = fused.get(item, 0) + 1 / (60 + rank)
for item, score in sorted(fused.items(), key=lambda pair: -pair[1]):
    print(item, f"{score:.6f}")

A 0.032522
B 0.032266
C 0.016129


A is second in one list and first in the other: 1/62 + 1/61 = 0.032522. B is first once but third in the other, so it
scores a little less; C is in one list only. Fusion rewards agreement. The same loop as a function:

In [13]:
def rrf(lists, c=60):
    scores = {}
    for ranked in lists:
        for rank, chunk_id in enumerate(ranked, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (c + rank)
    return sorted(scores, key=lambda chunk_id: -scores[chunk_id])

print(rrf([keyword, vector]))

['A', 'B', 'C']


Part B showed one question BM25 misses entirely, *What is the original meaning of the word Norman?* Fuse its two lists
and print where each of the top five came from:

In [14]:
norman = next(i for i, q in enumerate(questions) if "original meaning" in q["question"])
lists = [dense_lists[norman], bm25_lists[norman]]

def rank_in(ids, chunk_id):
    return ids.index(chunk_id) + 1 if chunk_id in ids else None

for chunk_id in rrf(lists)[:5]:
    ranks = [rank_in(ids, chunk_id) for ids in lists]
    score = sum(1 / (60 + rank) for rank in ranks if rank)
    known = "  <- known source" if src[chunk_id] in questions[norman]["known_support"] else ""
    print(f"{chunk_id}  vector {ranks[0]} | BM25 {ranks[1]} | RRF {score:.4f}{known}")

S02-P01-C00  vector 3 | BM25 3 | RRF 0.0317
S02-P00-C00  vector 2 | BM25 5 | RRF 0.0315
S02-P01-C02  vector 6 | BM25 6 | RRF 0.0303
S02-P02-C00  vector 1 | BM25 None | RRF 0.0164  <- known source
S26-P00-C00  vector None | BM25 1 | RRF 0.0164


The known source `S02-P02-C00` is first in the vector list and missing from the BM25 list, so it gets one contribution, 1/61 = 0.0164. Three chunks in both lists' top six score 0.0303 to 0.0317 and push it to fourth: when one list misses the answer, agreement works against it.

### Checkpoint C

| You should see | Our run |
|---|---|
| The example | A 0.032522, B 0.032266, C 0.016129; `rrf()` gives `['A', 'B', 'C']` |
| The Norman question | `S02-P02-C00` fourth with 0.0164, behind 0.0317, 0.0315 and 0.0303 |

---

## Part D: Compare hybrid search with the baseline, question by question

Part D measures the fused list against the vector list on the same 30 questions.

Fuse every question's two lists and count hit@1 and hit@3:

In [15]:
rrf_lists = [rrf([dense, keyword]) for dense, keyword in zip(dense_lists, bm25_lists)]
rrf_ranks = [first_hit(ids, q) for ids, q in zip(rrf_lists, questions)]
for name, ranks in [("vector", dense_ranks), ("hybrid", rrf_ranks)]:
    print(f"{name:6}  hit@1 {sum(hits(ranks, 1))} of 30 | hit@3 {sum(hits(ranks, 3))} of 30")

vector  hit@1 29 of 30 | hit@3 30 of 30
hybrid  hit@1 27 of 30 | hit@3 29 of 30


Two totals hide which questions moved. A **paired** comparison puts each question in one of four cells: passes with
both, passes only after the change (gained), passes only before (lost), or fails with both. Print the moved ones:

In [16]:
def paired(before, after):
    pairs = list(zip(before, after))
    return {"both": sum(b and a for b, a in pairs), "neither": sum(not b and not a for b, a in pairs),
            "gained": [i for i, (b, a) in enumerate(pairs) if a and not b],
            "lost": [i for i, (b, a) in enumerate(pairs) if b and not a]}

def show(before_ranks, after_ranks, after_name):
    for k in (3, 1):
        table = paired(hits(before_ranks, k), hits(after_ranks, k))
        print(f"hit@{k}: both {table['both']} | gained {len(table['gained'])} | lost {len(table['lost'])}"
              f" | neither {table['neither']}")
        for i in table["gained"] + table["lost"]:
            moved = "gained" if i in table["gained"] else "lost  "
            print(f"   {moved} vector rank {before_ranks[i]} | {after_name} rank {after_ranks[i]} | {questions[i]['question'].strip()}")

show(dense_ranks, rrf_ranks, "hybrid")

hit@3: both 29 | gained 0 | lost 1 | neither 0
   lost   vector rank 1 | hybrid rank 4 | What is the original meaning of the word Norman?
hit@1: both 26 | gained 1 | lost 3 | neither 0
   gained vector rank 2 | hybrid rank 1 | Who was the Super Bowl 50 MVP?
   lost   vector rank 1 | hybrid rank 2 | In what country is Normandy located?
   lost   vector rank 1 | hybrid rank 4 | What is the original meaning of the word Norman?
   lost   vector rank 1 | hybrid rank 3 | What measure of a computational problem broadly defines the inherent difficulty of the solution?


At hit@3 hybrid search keeps 29 questions and loses the Norman question. At hit@1 it gains the MVP question, BM25's one win, and loses three questions the vector search had right.

Is that split evidence, or chance? The [sign test](https://en.wikipedia.org/wiki/Sign_test) assumes the change makes
no difference, so each question that moved was a coin flip between gained and lost. Its p-value is the chance of a
split at least as uneven as yours. Questions that did not move are left out:

In [17]:
from math import comb

def sign_test(gains, losses):
    n = gains + losses
    tail = sum(comb(n, i) for i in range(min(gains, losses) + 1)) / 2 ** n
    return min(1.0, 2 * tail)

for k in (3, 1):
    table = paired(hits(dense_ranks, k), hits(rrf_ranks, k))
    gains, losses = len(table["gained"]), len(table["lost"])
    print(f"hit@{k}: {gains} gained, {losses} lost | p = {sign_test(gains, losses):.3f}")

hit@3: 0 gained, 1 lost | p = 1.000
hit@1: 1 gained, 3 lost | p = 0.625


p = 0.625 for 1 gained against 3 lost: a split like that is common when a change makes no difference. At hit@3, one lost question gives p = 1.000. With four moved questions the test cannot show that hybrid search is worse, and nothing shows it is
better. A change has to earn its place, and this one adds a second index to build and keep in step, for no gain here.
Keep the vector search as the baseline.

### Checkpoint D

| You should see | Our run |
|---|---|
| Totals | vector hit@1 29, hit@3 30; hybrid hit@1 27, hit@3 29 |
| Paired, hit@3 | both 29, gained 0, lost 1 (Norman) |
| Paired, hit@1 | both 26, gained 1 (MVP), lost 3 (Normandy, Norman, the computational measure) |
| Sign test | hit@3 p = 1.000; hit@1 p = 0.625 |

---

## Part E: Rerank the candidates with a chat model (self-study)

Part E adds a second stage after the search: a **reranker** reads the question with each candidate and reorders the
candidates. A chunk missing from the ten stays missing.

In production the reranker is usually a cross-encoder, a model that scores one question and passage pair at a time
(for example [Cohere Rerank](https://docs.cohere.com/docs/rerank-overview)), behind a vendor API or a model download.
With only a chat API, you ask the chat model to rank the whole list in one call (listwise reranking, as in
[RankGPT](https://arxiv.org/abs/2304.09542)).

The model comes from `lc_chat_model("RERANK_MODEL")`. Its reply is a Pydantic class with one field, the IDs in order;
`include_raw=True` also returns the raw message, which holds the token counts:

In [18]:
from pydantic import BaseModel
from shared.settings import lc_chat_model

class Ranking(BaseModel):
    ranking: list[str]

RULES = ("Rank the passages by how directly they answer the question."
         ' Reply in JSON shaped like {"ranking": ["ID", ...]}, with every passage ID once, best first.')
method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
reranker = lc_chat_model("RERANK_MODEL", settings).with_structured_output(Ranking, method=method, include_raw=True)
print("rerank model:", settings["RERANK_MODEL"], "| JSON_MODE:", settings["JSON_MODE"])

rerank model: gpt-4o-mini | JSON_MODE: json_schema


Send the recovery question with its ten vector candidates as `[ID] text` lines, the format the answer step reads:

In [19]:
candidates = lc_pipeline.retrieve(store, recovery, k=10)
passages = "\n".join(lc_pipeline.evidence(candidates))
start = time.perf_counter()
reply = reranker.invoke([("system", RULES), ("user", f"Question: {recovery}\nPassages:\n{passages}")])
seconds = time.perf_counter() - start

order = reply["parsed"].ranking
usage = reply["raw"].usage_metadata
print("vector:  ", [document.id for document, _ in candidates][:5])
print("reranked:", order[:5])
print(f"tokens: {usage['input_tokens']} in, {usage['output_tokens']} out | {seconds:.2f} s")

vector:   ['H01', 'H04', 'H02', 'H05', 'H03']
reranked: ['H02', 'H01', 'H04', 'H05', 'H03']
tokens: 561 in, 54 out | 1.14 s


The reranker puts H02 first: it read *expire after 30 minutes* as the answer to *stop working*, which neither search could. It read 561 tokens, the rules and the ten candidates, and took 1.14 seconds.

The IDs in the reply are text the model wrote. The JSON schema checks the shape, a list of strings, not which strings:
a reply can leave a candidate out, repeat one, or name one that was not a candidate. Check that it holds exactly the
candidate IDs, and keep the vector order when it does not:

In [20]:
ids = [document.id for document, _ in candidates]
print("exactly the candidate IDs:", sorted(order) == sorted(ids))

exactly the candidate IDs: True


### Checkpoint E

| You should see | Our run |
|---|---|
| Recovery question | vector `H01`, `H04`, `H02`, ...; reranked `H02`, `H01`, `H04`, ...; 561 tokens in, about a second |
| The ID check | `exactly the candidate IDs: True` |

---

## Part F: Rerank the 30 questions and count the cost (self-study)

Part F reranks every development question, checks every reply, and prices the new stage.

Part E's steps go into one function. It returns the order to use, the reply's order, whether the reply passed the ID
check, and the call's tokens and seconds:

In [21]:
def rerank(question, found):
    ids = [document.id for document, _ in found]
    user = f"Question: {question}\nPassages:\n" + "\n".join(lc_pipeline.evidence(found))
    start = time.perf_counter()
    reply = reranker.invoke([("system", RULES), ("user", user)])
    seconds = time.perf_counter() - start
    returned = reply["parsed"].ranking
    accepted = sorted(returned) == sorted(ids)
    usage = reply["raw"].usage_metadata
    return {"order": returned if accepted else ids, "returned": returned, "accepted": accepted,
            "input_tokens": usage["input_tokens"], "output_tokens": usage["output_tokens"], "seconds": seconds}

reranked = [rerank(q["question"], found) for q, found in zip(questions, dense_hits)]
print("replies that passed the ID check:", sum(r["accepted"] for r in reranked), "of 30")

replies that passed the ID check: 29 of 30


Print every reply that failed the check, with what was wrong:

In [22]:
for q, ids, r in zip(questions, dense_lists, reranked):
    if not r["accepted"]:
        returned = r["returned"]
        print(q["question"].strip())
        print("   left out:", sorted(set(ids) - set(returned)),
              "| not a candidate:", sorted(set(returned) - set(ids)),
              "| repeated:", sorted({i for i in returned if returned.count(i) > 1}))

Over how many species of trees can be found in the Saxon Garden?
   left out: ['S01-P02-C00', 'S16-P00-C02'] | not a candidate: ['S02-P00-C02'] | repeated: []


One reply of 30 failed. For the Saxon Garden question it left out two candidates and named `S02-P00-C02`, a chunk from the Normans article. That ID is real, so a check against the corpus would pass it; check against the candidates. Replies vary between runs, so ask the Saxon Garden question five more times with the same
candidates:

In [23]:
saxon = next(i for i, q in enumerate(questions) if "Saxon Garden" in q["question"])
for run in range(1, 6):
    r = rerank(questions[saxon]["question"], dense_hits[saxon])
    print(f"run {run}: passed {r['accepted']} | left out {sorted(set(dense_lists[saxon]) - set(r['returned']))}"
          f" | not a candidate {sorted(set(r['returned']) - set(dense_lists[saxon]))}")

run 1: passed False | left out ['S01-P02-C00', 'S16-P00-C02'] | not a candidate ['S02-P00-C02']


run 2: passed True | left out [] | not a candidate []


run 3: passed True | left out [] | not a candidate []


run 4: passed False | left out ['S01-P02-C00', 'S16-P00-C02'] | not a candidate ['S02-P00-C02']


run 5: passed False | left out ['S01-P02-C00', 'S16-P00-C02'] | not a candidate ['S02-P00-C02']


Three of the five replies failed the same way, with `S02-P00-C02` again; two passed. Count failures over several runs, and keep the check on every call.

Now the paired table, reranked against the vector order, with the same `show()` as Part D:

In [24]:
rerank_ranks = [first_hit(r["order"], q) for r, q in zip(reranked, questions)]
show(dense_ranks, rerank_ranks, "reranked")
table = paired(hits(dense_ranks, 1), hits(rerank_ranks, 1))
print(f"sign test at hit@1: p = {sign_test(len(table['gained']), len(table['lost'])):.3f}")

hit@3: both 30 | gained 0 | lost 0 | neither 0
hit@1: both 28 | gained 1 | lost 1 | neither 0
   gained vector rank 2 | reranked rank 1 | Who was the Super Bowl 50 MVP?
   lost   vector rank 1 | reranked rank 2 | What measure of a computational problem broadly defines the inherent difficulty of the solution?
sign test at hit@1: p = 1.000


hit@3 does not move: 30 of 30 both ways. At hit@1 the reranker gains the MVP question and loses the computational measure question: 29 both ways, p = 1.000. Read the lost question: what did the reranker put first, and what does the known source say?

In [25]:
text = {record["chunk_id"]: record["text"] for record in corpus}
for i in table["lost"]:
    first, known = reranked[i]["order"][0], dense_lists[i][dense_ranks[i] - 1]
    print(questions[i]["question"].strip())
    print(f"   reranked first {first}: {text[first][:150]} ...")
    print(f"   known source  {known}: {text[known][:150]} ...")

What measure of a computational problem broadly defines the inherent difficulty of the solution?
   reranked first S04-P00-C00: Computational complexity theory is a branch of the theory of computation in theoretical computer science that focuses on classifying computational pro ...
   known source  S04-P01-C00: A problem is regarded as inherently difficult if its solution requires significant resources, whatever the algorithm used. The theory formalizes this  ...


The reranker chose `S04-P00-C00`, which repeats the question's words *inherent difficulty* and defines the field, over `S04-P01-C00`, which holds the answer: *if its solution requires significant resources*. Shared words can mislead a reranker too.

Last, the cost. The search times come from Part B, the reranker's from its replies; the chat prices in `models.env`
apply because `RERANK_MODEL` is the chat model here. The last line counts the evidence the answer step reads:

In [26]:
import tiktoken
from shared.settings import chat_url

n = len(questions)
tokens_in = sum(r["input_tokens"] for r in reranked)
tokens_out = sum(r["output_tokens"] for r in reranked)
dollars = (tokens_in * float(settings["CHAT_PRICE_INPUT"]) + tokens_out * float(settings["CHAT_PRICE_OUTPUT"])) / 1e6
encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])
evidence = sum(len(encoding.encode("\n".join(lc_pipeline.evidence(found[:3])))) for found in dense_hits)

print(f"vector search  {1000 * vector_seconds / n:.1f} ms per question (one embedding call)")
print(f"BM25 search    {1000 * bm25_seconds / n:.1f} ms per question (no call)")
print(f"rerank         {1000 * sum(r['seconds'] for r in reranked) / n:.0f} ms per question | {tokens_in / n:.0f} tokens in,"
      f" {tokens_out / n:.0f} out | {1000 * dollars / n:.2f} US dollars per 1,000 questions")
print(f"answer step reads {evidence / n:.0f} tokens of evidence per question")
print(f"\n{settings['RERANK_MODEL']} at {chat_url(settings)}")

vector search  219.5 ms per question (one embedding call)
BM25 search    0.1 ms per question (no call)
rerank         1179 ms per question | 961 tokens in, 74 out | 0.19 US dollars per 1,000 questions
answer step reads 281 tokens of evidence per question

gpt-4o-mini at https://api.openai.com/v1


The vector search takes about 220 ms per question, nearly all of it the embedding call, and BM25 0.1 ms. The reranker takes 1,179 ms and reads 961 tokens, more than three times the 281 tokens of evidence the answer reads. At 0.19 US dollars per 1,000 questions it costs little money, but it adds more than a second to every answer.

### Checkpoint F

| You should see | Our run |
|---|---|
| ID check | 29 of 30 passed; the Saxon Garden reply named `S02-P00-C02` and failed again in 3 of 5 repeats |
| Paired, hit@3 and hit@1 | hit@3: both 30; hit@1: gained 1 (MVP), lost 1 (the computational measure), p = 1.000 |
| Cost per question | vector about 220 ms, BM25 0.1 ms; rerank about 1,179 ms, 961 tokens in, 74 out, 0.19 US dollars per 1,000 questions |

---

## Part G: Try both changes on Atlas questions and decide (self-study)

Part G tests the two changes on questions worded unlike their answers, then writes the decision.

The development questions were written from their paragraphs and share many words with them, so the vector search
already ranks a known source first for 29 of 30. Customers word questions their own way. Labs 01 and 04 wrote five
Atlas questions, each with the policy that answers it:

In [27]:
atlas = [  # written in Labs 01 and 04, each with the policy that answers it
    {"question": recovery, "known_support": ["H02"]},
    {"question": "How long does an Atlas reset link remain valid?", "known_support": ["H02"]},
    {"question": "Who can export all reports in Atlas?", "known_support": ["H03"]},
    {"question": "Can temporary Atlas accounts reset by email?", "known_support": ["H01"]},
    {"question": "Do I need the mobile app to reset my Atlas password?", "known_support": ["H01"]},
]
atlas_reranked = []
for q in atlas:
    found = lc_pipeline.retrieve(store, q["question"], k=10)
    ids = [document.id for document, _ in found]
    fused = rrf([ids, keyword_search(q["question"])])
    r = rerank(q["question"], found)
    atlas_reranked.append(r)
    check = "passed" if r["accepted"] else f"failed, left out {sorted(set(ids) - set(r['returned']))}"
    print(f"{q['known_support'][0]} | vector {ids[0]} | hybrid {fused[0]} | reranked {r['order'][0]}"
          f" | reply began {r['returned'][0]}, {check} | {q['question']}")

H02 | vector H01 | hybrid H01 | reranked H02 | reply began H02, passed | When does the password recovery email stop working for Atlas?


H02 | vector H02 | hybrid H02 | reranked H02 | reply began H02, passed | How long does an Atlas reset link remain valid?


H03 | vector H03 | hybrid H03 | reranked H03 | reply began H03, passed | Who can export all reports in Atlas?


H01 | vector H01 | hybrid H01 | reranked H01 | reply began H01, passed | Can temporary Atlas accounts reset by email?


H01 | vector H04 | hybrid H04 | reranked H04 | reply began H01, failed, left out ['H03'] | Do I need the mobile app to reset my Atlas password?


The vector search and hybrid search put the right policy first for 3 of 5 and miss the same two. The reranker fixes the recovery question, the paraphrase both searches miss. For the mobile-app question its reply began with H01, the right policy, but left out H03, so the ID check kept the vector order (Challenge task 1 repairs such replies).

### Write the decision

A decision names the change, the questions it was measured on, what moved and what it costs:

| Change | Paired, 30 development questions | Right first, 5 Atlas questions | Cost per question | Decision |
|---|---|---|---|---|
| Hybrid search (BM25 + RRF) | hit@3: lost 1; hit@1: gained 1, lost 3 (p = 0.625) | 3 of 5 (vector 3 of 5) | a second index; 0.1 ms | Keep the vector search |
| Reranking the vector top 10 | hit@3: no change; hit@1: gained 1, lost 1 (p = 1.000); 1 reply of 30 failed the ID check | 4 of 5 | 961 tokens, 1,179 ms, 0.19 US dollars per 1,000 questions | Not yet; test on more customer questions |

As one line: *Keep the vector search. Hybrid search lost a question and gained none at hit@3; reranking gained as many
as it lost, for 961 tokens and 1.2 seconds more per question. Its one Atlas gain is a
reason to collect more customer questions (Lab 06), not to ship it.*

### Checkpoint G

| You should see | Our run |
|---|---|
| Atlas questions | right first: vector 3 of 5, hybrid 3 of 5, reranked 4 of 5; the mobile-app reply began with H01 and failed, left out `['H03']` |

---

## Part H: Keep the pieces in `shared/hybrid.py` (self-study)

Part H moves this lab's functions into one module for the labs after this one: the BM25 index and search of Part B,
`rrf()`, `paired()`, `sign_test()` and `rerank()`, with its rules and `Ranking` class. Read it:

In [28]:
import inspect
from shared import hybrid

print(inspect.getsource(hybrid))

"""Hybrid search, reranking and paired comparisons: the steps Lab 08 builds by hand, kept for the labs after it.

bm25_index() and bm25_search() rank records by the words they share with a question (BM25, through bm25s, set up
like the hand version: every word, no stop-word list, k1=1.2, b=0.75). rrf() fuses ranked ID lists by reciprocal
rank. rerank() asks RERANK_MODEL to order the candidates and keeps the search order unless the reply holds exactly the
candidate IDs. paired() and sign_test() compare a change with the baseline question by question.
"""
from __future__ import annotations

import time
from math import comb

import bm25s
from pydantic import BaseModel

from shared.settings import lc_chat_model

WORDS = r"(?u)\b\w+\b"      # every word, one letter or more, as terms() in Lab 08

RERANK_RULES = (
    "Rank the passages by how directly they answer the question."
    ' Reply in JSON shaped like {"ranking": ["ID", ...]}, with every passage ID once, best first.'
)


class Ranki

Use it as a later lab will, and check it against what you built:

In [29]:
index = hybrid.bm25_index(corpus)
module_bm25 = [hybrid.bm25_search(index, q["question"]) for q in questions]
module_rrf = [hybrid.rrf([dense, keyword]) for dense, keyword in zip(dense_lists, module_bm25)]
print("same BM25 lists:", sum(a == b for a, b in zip(module_bm25, bm25_lists)), "of 30",
      "| same fused lists:", sum(a == b for a, b in zip(module_rrf, rrf_lists)), "of 30")
print("sign test, 1 gained and 3 lost:", hybrid.sign_test(1, 3), "| by hand:", sign_test(1, 3))
r = hybrid.rerank(recovery, candidates, settings)
print("recovery question reranked:", r["order"][:3], "| passed the ID check:", r["accepted"])

same BM25 lists: 30 of 30 | same fused lists: 30 of 30
sign test, 1 gained and 3 lost: 0.625 | by hand: 0.625


recovery question reranked: ['H02', 'H01', 'H04'] | passed the ID check: True


The module gives the same BM25 and fused lists for 30 of 30 questions and the same p-value, and puts H02 first for the recovery question. From here on the labs call these functions by name.

### Checkpoint H

| You should see | Our run |
|---|---|
| Module results | same BM25 lists 30 of 30, same fused lists 30 of 30; p = 0.625 both ways; `['H02', 'H01', 'H04']`, passed |

---

## What you can do now

- Score chunks with BM25 by hand and with `bm25s`, and say which words a score came from.
- Fuse two ranked lists with reciprocal rank fusion, and say why it adds ranks, not scores.
- Compare a change with the baseline in a paired table, name the questions it gained and lost, and run a sign test.
- Rerank candidates with a chat model and reject a reply that is not exactly the candidate IDs.
- Price a retrieval stage in tokens, dollars and seconds per question.
- Write a decision from the paired results and the cost, and say what evidence would change it.

## Going further: diverse evidence with maximal marginal relevance

Two of the three chunks an answer reads can say nearly the same thing. **Maximal marginal relevance** (MMR) picks each
next chunk by its similarity to the question minus its similarity to the chunks already picked:

In [30]:
mmr_lists = [[d.id for d in store.max_marginal_relevance_search(q["question"], k=3, fetch_k=10)] for q in questions]
mmr_ranks = [first_hit(ids, q) for ids, q in zip(mmr_lists, questions)]
print(f"MMR hit@1 {sum(hits(mmr_ranks, 1))} of 30 | hit@3 {sum(hits(mmr_ranks, 3))} of 30"
      f" | top three differ from the vector top three: {sum(a != b[:3] for a, b in zip(mmr_lists, dense_lists))} of 30")

MMR hit@1 29 of 30 | hit@3 30 of 30 | top three differ from the vector top three: 30 of 30


MMR changes the top three of all 30 questions, and hit@1 (29) and hit@3 (30) stay the same: it keeps the most similar chunk first. hit@k cannot see what MMR changes; judge it on the answers, with Lab 06's measures. Reranking the fused
list can be tested the same way, paired against the vector baseline.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is optional. For each task, keep the
printed results and your answer to **Explain**.

1. **Repair a rejected ranking.** The ID check throws a whole reply away, even when one unimportant candidate is
   missing. A common repair keeps the reply's order for the IDs that are candidates, drops repeats and IDs that were
   not candidates, and adds the missing candidates at the end in the vector order.

   - Complete `repair()` under **Task 1**. The cell checks it on a made-up reply, then repairs every reply that failed
     in Parts F and G and prints the first ID before and after.

   **Check:** the made-up reply gives `['B', 'A', 'C', 'D']`. Our worked solution moved the mobile-app question's first ID from H04 to H01.

   **Explain:** was the changed first ID right? What could a repair hide that a rejection shows?

2. **Optional: check whether the candidate order sways the reranker.** Under **Task 2**, shuffle the recovery
   question's ten candidates five times (`random.Random(seed)`) and rerank each order.

   **Check:** how many of the five put H02 first. Our worked solution put H02 first in 5 of 5 orders, with H02 listed anywhere from second to tenth.

   **Explain:** what would you conclude if H02 came first only when it was listed first?

**Task 1.** Your `repair()`, then the check:

In [31]:
def repair(returned, candidates):
    # TODO: the candidates in the reply's order (each once, no IDs that are not candidates),
    #       then the candidates the reply left out, in their own order
    return None

if repair(["B", "X", "B"], ["A", "B", "C", "D"]) is None:
    print("⚠️ Complete repair(), then run the cell again.")
else:
    print("made-up reply:", repair(["B", "A", "X", "B"], ["A", "B", "C", "D"]))
    failed = [(q, r) for q, r in zip(questions + atlas, reranked + atlas_reranked) if not r["accepted"]]
    for q, r in failed:
        fixed = repair(r["returned"], r["order"])
        print(f"{q['known_support']} | first before {r['order'][0]} | after {fixed[0]} | {q['question'].strip()}")

⚠️ Complete repair(), then run the cell again.


**Task 2 (optional).** Five shuffled orders of the recovery question's candidates:

In [32]:
import random

firsts = []
for seed in range(5):
    shuffled = list(candidates)
    # TODO: shuffle `shuffled` with random.Random(seed), rerank it with rerank(recovery, shuffled),
    #       and append the first ID of the order to firsts

if not firsts:
    print("⚠️ Shuffle and rerank, then run the cell again.")
else:
    print("first ID per order:", firsts, "| H02 first:", firsts.count("H02"), "of", len(firsts))

⚠️ Shuffle and rerank, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `bm25s` scores in another order than the hand version (H02 last for the recovery question) | `bm25s` defaults: English stop words dropped, one-letter words ignored, `k1=1.5` | Pass `token_pattern=WORDS, stopwords=None` and `k1=1.2, b=0.75`, as `hybrid.bm25_index()` does |
| `ValueError: k of 10 is larger than the number of available scores, which is 5 (corpus size should be larger than top-k)...` | `retrieve(k=10)` on the five-policy index | Ask for at most as many results as the index holds |
| `OpenAIInvalidRequestError: Error code: 400 ... 'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.` | `JSON_MODE=json_object` with a rule that never says JSON | Keep the word JSON in the rule, as `RULES` does |
| `OpenAIInvalidRequestError: Error code: 400 ... Unrecognized request argument supplied: reasoning_effort` on the first rerank call | `REASONING_EFFORT` set for a reasoning chat model while `RERANK_MODEL` is still `gpt-4o-mini` | Switch `CHAT_MODEL`, `JUDGE_MODEL` and `RERANK_MODEL` together |

## Resources

- [`bm25s`](https://github.com/xhluca/bm25s): the BM25 package, its tokenizer options and scoring variants
- [Okapi BM25](https://en.wikipedia.org/wiki/Okapi_BM25): the formula, IDF, `k1` and `b`
- [Cormack, Clarke and Büttcher (2009)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf): reciprocal rank
  fusion and `c = 60`
- [Elasticsearch: reciprocal rank fusion](https://www.elastic.co/docs/reference/elasticsearch/rest-apis/reciprocal-rank-fusion)
  and [Qdrant: hybrid queries](https://qdrant.tech/documentation/concepts/hybrid-queries/): hybrid search inside a
  search engine and a vector database
- [Sun and others (2023), RankGPT](https://arxiv.org/abs/2304.09542): listwise reranking with a chat model
- [Cohere: rerank overview](https://docs.cohere.com/docs/rerank-overview): a production cross-encoder reranker
- [LangChain v1 migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1): where the legacy
  retrievers moved

Next: [Lab 09: How to answer follow-up questions with retrieval](../09_Follow_Up_Questions/README.md). Back to
[the lab list](../README.md).